# 01 Quantitative Enrollment Analysis (Public Version)

This notebook is a clean public version of the quantitative part of the project. It uses the synthetic public CSV at `data/sample_anonymized_enrollment.csv` and focuses on aggregate enrollment, course, and age trends.

The private source records are not included here.

## Public Dataset and Privacy Note

The CSV is synthetic/anonymized. It is designed to let readers rerun the workflow without exposing any personal record. Cells below avoid row-by-row previews and report only aggregate outputs.

## Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "sample_anonymized_enrollment.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import (
    add_canonical_course_column,
    coerce_numeric_age,
    normalize_columns,
)
from src.enrollment_analysis import (
    DEFAULT_COURSE_ORDER,
    age_range_distribution,
    age_statistics_by_year,
    annual_enrollment_counts,
    course_year_matrix,
)

## Load `data/sample_anonymized_enrollment.csv`

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "sample_anonymized_enrollment.csv"

df = pd.read_csv(DATA_PATH)
df = normalize_columns(df)
df = add_canonical_course_column(df, "course", output_column="course_standard")
df = coerce_numeric_age(df, "age", output_column="age_numeric")

record_summary = pd.DataFrame(
    {
        "metric": ["records", "registry_years", "course_groups"],
        "value": [
            len(df),
            df["registry_year"].nunique(),
            df["course_standard"].nunique(),
        ],
    }
)
record_summary

## Annual Enrollment Counts

In [ ]:
annual_counts = annual_enrollment_counts(df, "registry_year")
annual_counts

## Course-Year Matrix

In [ ]:
course_matrix = course_year_matrix(
    df,
    year_column="registry_year",
    course_column="course_standard",
    course_order=DEFAULT_COURSE_ORDER,
)
course_matrix

## Age Statistics

In [ ]:
age_stats = age_statistics_by_year(
    df,
    year_column="registry_year",
    age_column="age_numeric",
).round(2)
age_stats

## Age Range Distribution

In [ ]:
age_ranges = age_range_distribution(
    df,
    year_column="registry_year",
    age_column="age_numeric",
)
age_ranges

## Key Findings

- The public sample is balanced at 30 records for each registry year from 1910 through 1915.
- First-year enrollment is the largest course group in most years of the sample.
- Median age stays near 17-18 across the period, with most records in the 16-17 and 18-24 bands.
- These findings describe the public sample only; the project page reports notebook-derived aggregate findings from the source workflow.

## Reproducibility Note

Run this notebook from the repository root so `data/sample_anonymized_enrollment.csv` is found. The command-line runner generates the committed aggregate outputs with:

```bash
python src/run_public_analysis.py --input data/sample_anonymized_enrollment.csv --output outputs
```